In [88]:
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd
from tqdm import tqdm
import numpy as np

In [89]:

# 스크립트 기준 경로 고정
base_dir = Path("./..")
data_dir = base_dir / 'data'
img_dir = base_dir / 'imgs'

# 1) 데이터 로드
csv_path = data_dir / 'extraction.csv'
df = pd.read_csv(csv_path, encoding='cp949')
# 필수 컬럼 확인 (xpos, ypos, call_date)
for col in ['xpos', 'ypos', 'call_date']:
    if col not in df.columns:
        raise KeyError(f"필수 컬럼 누락: {col}")
df['call_date'] = pd.to_datetime(df['call_date'], format='%Y-%m-%d %H')

X = df[['call_date']].to_numpy(dtype=float)
n = X.shape[0]
print(f"X shape: {X.shape}")

X shape: (369227, 1)


In [90]:
start_time = df['call_date'].min()
end_time = df['call_date'].max()
total_hours = int((end_time - start_time).total_seconds() // 3600) + 1
print(f"데이터 시간 범위: {start_time} ~ {end_time} ({total_hours} 시간)")

데이터 시간 범위: 2024-10-01 00:00:00 ~ 2025-03-31 23:00:00 (4368 시간)


In [91]:
demand_arr = np.zeros(total_hours, dtype=int)
for i in tqdm(range(n)):
    call_time = df.loc[i, 'call_date']
    hour_index = int((call_time - start_time).total_seconds() // 3600)
    demand_arr[hour_index] += 1
mean_daily_demand = demand_arr.reshape(-1, 24)
mean_daily_demand = mean_daily_demand.mean(axis=0)
# 시각화
plt.figure(figsize=(10, 6))
plt.plot(range(24), mean_daily_demand, marker='o')
plt.title('Demand over Time')
plt.xlabel('time (hours)')
plt.ylabel('demands')
plt.grid()
plt.savefig(img_dir / 'demand_over_time.png')
plt.close()

100%|██████████| 369227/369227 [00:04<00:00, 81960.41it/s]


In [92]:
meteological_data_path = data_dir / 'meteorological_data.csv'
meteorological_df = pd.read_csv(meteological_data_path, encoding='cp949')
meteorological_df.fillna(0, inplace=True)
target_col = ["강수량(mm)", "적설(cm)"]

meteo_arr = np.zeros((total_hours), dtype=float)

for i in tqdm(range(total_hours)):
    current_time = start_time + pd.Timedelta(hours=i)
    meteo_row = meteorological_df[meteorological_df['일시'] == current_time.strftime('%Y-%m-%d %H:%M')]
    if not meteo_row.empty:
        for j, col in enumerate(target_col):
            meteo_arr[i] += meteo_row.iloc[0][col]

# 시각화
plt.figure(figsize=(10, 6))
plt.plot(range(total_hours), meteo_arr, marker='o', label='Meteorological Data')
plt.title('Meteorological Data over Time')
plt.xlabel('time (hours)')  
plt.ylabel('Value')
plt.grid()
plt.savefig(img_dir / 'meteorological_data_over_time.png')
plt.close()

print(f'not zero meteorological data count: {(meteo_arr != 0).sum()}')

100%|██████████| 4368/4368 [00:02<00:00, 1956.61it/s]


not zero meteorological data count: 167


In [93]:
print(f'demand_arr shape: {demand_arr.shape}', f'meteo_arr shape: {meteo_arr.shape}')

demand_arr shape: (4368,) meteo_arr shape: (4368,)


In [94]:
daily_meteo = meteo_arr.reshape(-1, 24)
daily_meteo[daily_meteo != 0] = 1  # 이진화
print(f'sum of daily_meteo: {daily_meteo.sum(axis=1)}')
daily_demand = demand_arr.reshape(-1, 24)

mean_not_meteo_demand = daily_demand[daily_meteo.sum(axis=1) == 0].mean(axis=0)
mean_meteo_demand = daily_demand[daily_meteo.sum(axis=1) != 0].mean(axis=0)

#시각화
plt.figure(figsize=(10, 6))
plt.plot(range(24), mean_not_meteo_demand, marker='o', label='No Meteorological Events')
plt.plot(range(24), mean_meteo_demand, marker='o', label='With Meteorological Events')
plt.title('Demand over Time by Meteorological Events') 
plt.xlabel('time (hours)')
plt.ylabel('demands')
plt.legend()
plt.grid()
plt.savefig(img_dir / 'demand_by_meteorological_events.png')
plt.close()

print(f'mean_not_meteo_demand: {mean_not_meteo_demand}')
print(f'mean_meteo_demand: {mean_meteo_demand}')


sum of daily_meteo: [ 0.  0.  9.  2.  0.  5. 13.  2.  0.  0.  0.  0.  0.  9.  8.  0.  0.  6.
  5.  2.  4. 20.  0.  0.  0.  0.  5.  5.  0.  0.  0.  5.  4.  0.  0.  0.
  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  2.  0.  0.  0.  0.
  0.  0.  4.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.
  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.
  0.  0.  0.  0.  0.  0.  0.  0.  0.  2.  9.  0.  0.  0.  0.  0.  0.  0.
  0.  0.  0.  0.  0.  0.  0.  0.  4.  0.  2.  0.  0.  0.  0.  4.  0.  0.
  0.  0.  0.  0.  0.  0.  0.  0.  3.  0.  0.  0.  0.  0.  0.  0.  0.  0.
  0.  0.  0.  0.  0.  0.  0.  0.  2.  6.  6.  4.  0.  0.  1.  0.  0.  0.
  0.  0.  0.  4.  6.  0.  1.  0.  0.  0.  0.  0.  0.  0.  0.  1.  2.  0.
  0.  0.]
mean_not_meteo_demand: [ 44.49324324  32.40540541  28.37162162  23.53378378  39.37162162
  56.89189189  70.02027027  80.2027027  130.44594595 149.60810811
 132.40540541 108.77027027  91.33783784 108.47297297 101.69594595
  98.63513514 107.5

In [95]:
percentage_diff = (mean_meteo_demand - mean_not_meteo_demand) / mean_not_meteo_demand * 100
print(np.mean(percentage_diff), np.std(percentage_diff))

7.887211522378055 3.7244368637481977


In [96]:
day_of_time = 24
month_of_day = 30

atom = int(day_of_time * month_of_day * 1.5)
winter = int(day_of_time * month_of_day * 2.5) + atom

atom_demands = demand_arr[:atom]
winter_demands = demand_arr[atom:winter]
spring_demands = demand_arr[winter:]
print(f'atom_demands shape: {atom_demands.shape}, winter_demands shape: {winter_demands.shape}, spring_demands shape: {spring_demands.shape}')

daily_atom_demands = atom_demands.reshape(-1, 24).mean(axis=0)
daily_winter_demands = winter_demands.reshape(-1, 24).mean(axis=0)
daily_spring_demands = spring_demands.reshape(-1, 24).mean(axis=0)

# 시각화
plt.figure(figsize=(10, 6))
plt.plot(range(24), daily_atom_demands, marker='o', label='fall')
plt.plot(range(24), daily_winter_demands, marker='o', label='winter')
plt.plot(range(24), daily_spring_demands, marker='o', label='spring')
plt.title('Seasonal Demand over Time')
plt.xlabel('time (hours)') 
plt.ylabel('demands')
plt.legend()
plt.grid()
plt.savefig(img_dir / 'seasonal_demand_over_time.png')
plt.close()

#평균 수요대비 계절별 수요 비율
mean_daily_demand = demand_arr.reshape(-1, 24).mean(axis=0)
ratio_atom = daily_atom_demands / mean_daily_demand * 100 -100.0
ratio_winter = daily_winter_demands / mean_daily_demand * 100 -100.0
ratio_spring = daily_spring_demands / mean_daily_demand * 100   -100.0
print(f' ratio_atom: {ratio_atom.mean()}, ratio_winter: {ratio_winter.mean()}, ratio_spring: {ratio_spring.mean()}')


atom_demands shape: (1080,), winter_demands shape: (1800,), spring_demands shape: (1488,)
 ratio_atom: 3.915621255156928, ratio_winter: 2.3116531075831612, ratio_spring: -5.638337734690297


In [79]:
# 정규화
xpos_min, xpos_max = df['xpos'].min(), df['xpos'].max()
ypos_min, ypos_max = df['ypos'].min(), df['ypos'].max()
print(f'xpos range: {xpos_min} ~ {xpos_max}, ypos range: {ypos_min} ~ {ypos_max}')
grid_size = 5000
df['xpos'] = (df['xpos'] - xpos_min)//grid_size
df['ypos'] = (df['ypos'] - ypos_min)//grid_size


xpos range: 330928 ~ 415467, ypos range: 183991 ~ 268517


In [80]:
demand_map = df.groupby(['xpos', 'ypos']).size().unstack(fill_value=0)
print(f'demand_map shape: {demand_map.shape}')
plt.figure(figsize=(8, 6))
plt.imshow(demand_map, cmap='hot', interpolation='nearest')
plt.title('Demand Heatmap')
plt.colorbar(label='Number of Demands')
plt.savefig(img_dir / 'demand_heatmap.png')
plt.close()

demand_map shape: (17, 17)


In [87]:
hot_point = [(35.5539, 129.3183), (36.196099, 127.080739)] 

for lat, lon in hot_point:
    lat = lat * 1000 - xpos_min
    lon = lon * 1000 - ypos_min
    grid_x = int(lat) // grid_size
    grid_y = int(lon) // grid_size
    print(f'Hot point at grid ({grid_x}, {grid_y})')


Hot point at grid (-60, -11)
Hot point at grid (-59, -12)
